# VitalDB train-only predictive feature selection

This notebook evaluates individual dynamic features for 30-second future-BIS prediction using patient-grouped Elastic Net stability, XGBoost held-out patient-block permutation stability, train-only correlations, and multi-method consensus. It does not run deep GRU/Attention retraining. Validation labels and held-out test data remain sealed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = Path('/content/VitalDB-Feature-Selection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR = DRIVE_PROJECT_ROOT / 'data/modeling/full'
GROUP_ANALYSIS_DIR = (
    DRIVE_PROJECT_ROOT / 'outputs/group_retraining_validation_only/analysis'
)
OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'outputs/predictive_feature_selection_30s'

TREE_DEVICE = 'cpu'  # Change to 'cuda' only when a GPU runtime is assigned.
COMPUTE_AUXILIARY_SHAP = False
STABILITY_ITERATIONS = 100
PROTECTED_CONTROL_FEATURES = ()  # Fill only after inspecting the external RL state.
RANDOM_SEED = 20260716
print({'dataset': str(DATASET_DIR), 'prior_analysis': str(GROUP_ANALYSIS_DIR), 'output': str(OUTPUT_DIR)})

## Repository and dependencies

The workflow is CPU-compatible. XGBoost can optionally use CUDA, while Elastic Net and correlation analysis remain CPU tasks.

In [ ]:
import os
import subprocess
import sys

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
CURRENT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
subprocess.run([
    sys.executable, 'scripts/install_colab_dependencies.py',
    '--requirements', 'requirements-colab.txt',
], check=True)
print('Selection code commit:', CURRENT_COMMIT)

## Pre-run plan and feature inventory

Only the existing train artifact is used for fitting. Static covariates are adjustment inputs and are not eligible dynamic candidates. `bis_error` is audited as a deterministic duplicate and excluded.

In [ ]:
import json

if not GROUP_ANALYSIS_DIR.is_dir():
    raise FileNotFoundError(f'Prior group analysis is missing: {GROUP_ANALYSIS_DIR}')
dataset_metadata = json.loads(
    (DATASET_DIR / 'dataset_metadata.json').read_text(encoding='utf-8')
)
full17 = [
    name for name in dataset_metadata['dynamic_feature_names'] if name != 'bis_error'
]
plan = {
    'data_scope': 'train split only',
    'target': '30-second future BIS',
    'patient_grouping': 'case_id for folds and subsampling',
    'dynamic_candidates': full17,
    'static_adjustment_only': dataset_metadata['static_feature_names'],
    'stability_iterations': STABILITY_ITERATIONS,
    'tree_device': TREE_DEVICE,
    'auxiliary_shap': COMPUTE_AUXILIARY_SHAP,
    'protected_control_features': list(PROTECTED_CONTROL_FEATURES),
    'maximum_frozen_retraining_candidates': 6,
}
print(json.dumps(plan, indent=2))

## Run train-only selectors

The prior group-ablation analysis is read-only context. It is not used to tune Elastic Net or XGBoost.

In [ ]:
command = [
    sys.executable, 'scripts/run_predictive_feature_selection.py',
    '--dataset-dir', str(DATASET_DIR),
    '--group-analysis-dir', str(GROUP_ANALYSIS_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--stability-iterations', str(STABILITY_ITERATIONS),
    '--tree-device', TREE_DEVICE,
    '--random-seed', str(RANDOM_SEED),
]
if COMPUTE_AUXILIARY_SHAP:
    command.append('--compute-shap')
if PROTECTED_CONTROL_FEATURES:
    command.extend([
        '--protected-control-features', ','.join(PROTECTED_CONTROL_FEATURES)
    ])
print('Running train-only selection:', ' '.join(command))
subprocess.run(command, check=True)

## Frozen candidate subsets and consensus evidence

In [ ]:
import pandas as pd
from IPython.display import display

candidate_subsets = json.loads(
    (OUTPUT_DIR / 'candidate_subsets.json').read_text(encoding='utf-8')
)
consensus = pd.read_csv(OUTPUT_DIR / 'consensus_feature_table.csv')
print(json.dumps(candidate_subsets, indent=2))
display(consensus)

## Train-only figures and report

In [ ]:
from IPython.display import Image, display

for figure_path in sorted((OUTPUT_DIR / 'figures').glob('*.png')):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))
report = (OUTPUT_DIR / 'predictive_feature_selection_report.md').read_text(
    encoding='utf-8'
)
print(report[:15_000])
print('Persistent selection outputs:', OUTPUT_DIR)